# Tarea 4
## Ejercicio 1

### Tlacaelel Jaime Flores Villaseñor

# Imports

In [13]:
import numpy as np
import os
from pathlib import Path
import hnswlib
import gc
from sklearn.feature_extraction.text import TfidfTransformer

# Carga de descriptores SIFT

In [ ]:

def load_sift_descriptors(folder_path: str, split: str):
    """
    Carga descriptores SIFT organizados train  y test.
    """
    base_path = Path(folder_path) / split
    X = []  # Lista para almacenar matrices de descriptores por imagen
    y = []  # Etiquetas numéricas
    
    # Obtener nombres de las clases (carpetas dentro de train/test)
    classes = sorted([d.name for d in base_path.iterdir() if d.is_dir()])
    class_to_idx = {name: i for i, name in enumerate(classes)}
    
    print(f"Cargando partición: {split}")
    
    for class_name in classes:
        
        features_dir = base_path / class_name / "features"
        
        if not features_dir.exists():
            print(f"Advertencia: No se encontró la carpeta features en {class_name}")
            continue
            
        # Iterar sobre los archivos .npy
        for npy_file in features_dir.glob("*.npy"):
            # Los descriptores SIFT son de 128 dimensiones
            descriptors = np.load(npy_file).astype(np.float32)
            X.append(descriptors)
            y.append(class_to_idx[class_name])
            
    return X, np.array(y), class_to_idx
            

In [3]:
X_train, y_train, mapping = load_sift_descriptors("data_tarea", "train")
X_test, y_test, _ = load_sift_descriptors("data_tarea", "test")

Cargando partición: train
Cargando partición: test


Tenemos  clases con 100 imagenes cada una en la carpeta de train, por lo que el tamaño de X train debe ser $15 \times 100 = 1500$

In [4]:
len(X_train)


1500

In [5]:
len(y_train)

1500

In [6]:
mapping

{'bedroom': 0,
 'coast': 1,
 'forest': 2,
 'highway': 3,
 'industrial': 4,
 'insidecity': 5,
 'kitchen': 6,
 'livingroom': 7,
 'mountain': 8,
 'office': 9,
 'opencountry': 10,
 'store': 11,
 'street': 12,
 'suburb': 13,
 'tallbuilding': 14}

In [8]:
def build_visual_vocabulary(data, k=10000, iterations=5):
    """
    Implementación de K-Means con HNSW.
    """
    n_samples, dim = data.shape
    
    print(f"Inicializando {k} centroides...")
    initial_indices = np.random.choice(n_samples, k, replace=False)
    centroids = data[initial_indices].copy()
    
    for i in range(iterations):
        print(f"Iteración {i+1}/{iterations} iniciada...")
        
        # Configuración del índice HNSW para los centroides
        # M: número de conexiones bidireccionales
        # ef_construction: tiempo/precisión en la construcción
        index = hnswlib.Index(space='l2', dim=dim)
        index.init_index(max_elements=k, ef_construction=200, M=16)
        index.add_items(centroids)
        index.set_ef(50) # Precisión en la búsqueda
        
        # Buscar el centroide más cercano para cada punto de data
        # labels: (n_samples, 1), contiene el índice del centroide asignado
        labels, _ = index.knn_query(data, k=1)
        labels = labels.flatten()
        
        # Mover los centroides al promedio de sus puntos asignados
        new_centroids = np.zeros_like(centroids)
        counts = np.zeros(k)
        
        # Vectorización parcial para calcular promedios
        for j in range(k):
            mask = (labels == j)
            if np.any(mask):
                new_centroids[j] = data[mask].mean(axis=0)
            else:
                # Si un centroide queda huérfano, se re-inicializa con un punto aleatorio
                new_centroids[j] = data[np.random.randint(n_samples)]
        
        centroids = new_centroids
        
        # Limpieza de memoria del índice en cada iteración
        del index
        gc.collect()
        
    return centroids

In [10]:
# Ejecución con la matriz concatenada de entrenamiento
X_train_stacked = np.vstack(X_train).astype(np.float32)
visual_vocabulary = build_visual_vocabulary(X_train_stacked)

Inicializando 10000 centroides...
Iteración 1/5 iniciada...
Iteración 2/5 iniciada...
Iteración 3/5 iniciada...
Iteración 4/5 iniciada...
Iteración 5/5 iniciada...


Guardar el vocabulario

In [11]:
np.save("visual_vocabulary_10k.npy", visual_vocabulary)

In [12]:
# LIBERAR MEMORIA
del X_train_stacked
gc.collect()


0

# Cuantización de Descriptores

definimos una función que transforme la lista de descriptores de cada imagen en un histograma de frecuencias.

In [ ]:
# Leer el vocabulario visual desde el archivo guardado
visual_vocabulary = np.load("visual_vocabulary_10k.npy")

In [14]:
def compute_bof_histograms(X_list, vocabulary, k=10000):
    """
    Transforma listas de descriptores en histogramas de palabras visuales.
    """
    dim = vocabulary.shape[1]
    
    # Configurar índice HNSW para búsqueda rápida de vecinos
    index = hnswlib.Index(space='l2', dim=dim)
    index.init_index(max_elements=k, ef_construction=200, M=16)
    index.add_items(vocabulary)
    index.set_ef(50)
    
    histograms = []
    
    print(f"Procesando {len(X_list)} imágenes...")
    for descriptors in X_list:
        # Encontrar el centroide más cercano para cada descriptor de la imagen
        labels, _ = index.knn_query(descriptors.astype(np.float32), k=1)
        
        # Construir el histograma de frecuencias minlength asegura que el histograma siempre tenga tamaño 10,000
        counts = np.bincount(labels.flatten(), minlength=k)
        histograms.append(counts)
        
    return np.array(histograms, dtype=np.float32)

# Ejecución para ambos conjuntos
bof_train_raw = compute_bof_histograms(X_train, visual_vocabulary)
bof_test_raw = compute_bof_histograms(X_test, visual_vocabulary)

Procesando 1500 imágenes...
Procesando 2985 imágenes...


los histogramas deben ponderarse mediante tf-idf y normalizarse.  La ponderación $tf-idf$ para una palabra visual $w$ en una imagen $d$ se define como:$$tf\text{-}idf(w, d) = tf(w, d) \times \log\left(\frac{N}{df(w)}\right)$$
Donde $N$ es el número total de imágenes y $df(w)$ es el número de imágenes que contienen la palabra $w$.  

In [15]:
tfidf = TfidfTransformer(norm='l2', use_idf=True)

# Ajustar el transformador SOLO con los datos de entrenamiento
bof_train = tfidf.fit_transform(bof_train_raw).toarray()

# Aplicar a los datos de prueba
bof_test = tfidf.transform(bof_test_raw).toarray()

print(f"Forma final de la matriz de entrenamiento: {bof_train.shape}")

Forma final de la matriz de entrenamiento: (1500, 10000)


In [16]:
np.save("bof_train_final.npy", bof_train)
np.save("bof_test_final.npy", bof_test)